import all packages and library 

In [4]:
import asyncio
import os
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import AzureOpenAIEmbeddings
from neo4j_graphrag.llm import AzureOpenAILLM
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import (
    FixedSizeSplitter,
)
import glob
from pathlib import Path 
from typing import List, Optional
from typing import Dict
from typing import TypedDict
from langgraph.graph import StateGraph, END
from neo4j_graphrag.retrievers import HybridRetriever

from langchain_text_splitters import RecursiveCharacterTextSplitter
from neo4j_graphrag.experimental.components.text_splitters.langchain import LangChainTextSplitterAdapter
from neo4j_graphrag.retrievers import VectorCypherRetriever
import json
import re


importing all files

In [5]:
directory=Path(r"C:\Users\JDas21\GRAPH-RAG\Sample Files")
pdf_files=[str(f) for f in directory.glob("*.pdf")]
print(pdf_files)

['C:\\Users\\JDas21\\GRAPH-RAG\\Sample Files\\Chen.pdf']


Define schema and Index Name for embedding 

In [6]:
INDEX_NAME = "chunk_embeddings"
node_types=[
"Article",
"Journal",
"Author",
"Institution",
"ClinicalStudy",
"RegistryRecord",
"StudyDesign",
"Cohort",
"Population",
"CharacteristicValue",
"EligibilityCriterion",
"Intervention",
"Drug",
"Dosing",
"Endpoint",
"AssessmentCriterion",
"OutcomeMeasure",
"ResultSummary",
"SafetyFinding",
"AdverseEvent",
"Biomarker",
"GenomicAlteration",
"SubgroupFinding"
]
relationship_types=["PUBLISHED_IN",
"HAS_AUTHOR",
"AFFILIATED_WITH",
"REPORTS",
"REGISTERED_AS",
"HAS_DESIGN",
"HAS_ELIGIBILITY",
"HAS_COHORT",
"HAS_POPULATION",
"HAS_CHARACTERISTIC",
"TESTS_INTERVENTION",
"USES_DRUG",
"HAS_DOSING",
"HAS_ENDPOINT",
"ASSESSED_BY",
"HAS_OUTCOME",
"REPORTED_FOR",
"SUMMARIZES",
"HAS_SAFETY_FINDING",
"HAS_SAFETY_EVENT",
"HAS_BIOMARKER",
"HAS_GENOMIC_ALTERATION",
"REPORTS_SUBGROUP_FINDING",
"BY_BIOMARKER",
"BY_GENOMIC_ALTERATION"
]
patterns = [
[
"Article",
"PUBLISHED_IN",
"Journal"
],
[
"Article",
"HAS_AUTHOR",
"Author"
],
[
"Author",
"AFFILIATED_WITH",
"Institution"
],
[
"Article",
"REPORTS",
"ClinicalStudy"
],
[
"ClinicalStudy",
"REGISTERED_AS",
"RegistryRecord"
],
[
"ClinicalStudy",
"HAS_DESIGN",
"StudyDesign"
],
[
"ClinicalStudy",
"HAS_ELIGIBILITY",
"EligibilityCriterion"
],
[
"ClinicalStudy",
"HAS_COHORT",
"Cohort"
],
[
"Cohort",
"HAS_POPULATION",
"Population"
],
[
"Population",
"HAS_CHARACTERISTIC",
"CharacteristicValue"
],
[
"ClinicalStudy",
"TESTS_INTERVENTION",
"Intervention"
],
[
"Intervention",
"USES_DRUG",
"Drug"
],
[
"Drug",
"HAS_DOSING",
"Dosing"
],
[
"ClinicalStudy",
"HAS_ENDPOINT",
"Endpoint"
],
[
"Endpoint",
"ASSESSED_BY",
"AssessmentCriterion"
],
[
"Endpoint",
"HAS_OUTCOME",
"OutcomeMeasure"
],
[
"OutcomeMeasure",
"REPORTED_FOR",
"Cohort"
],
[
"ClinicalStudy",
"SUMMARIZES",
"ResultSummary"
],
[
"ResultSummary",
"REPORTED_FOR",
"Cohort"
],
[
"ClinicalStudy",
"HAS_SAFETY_FINDING",
"SafetyFinding"
],
[
"ClinicalStudy",
"HAS_SAFETY_EVENT",
"AdverseEvent"
],
[
"AdverseEvent",
"REPORTED_FOR",
"Cohort"
],
[
"Population",
"HAS_BIOMARKER",
"Biomarker"
],
[
"Population",
"HAS_GENOMIC_ALTERATION",
"GenomicAlteration"
],
[
"ClinicalStudy",
"REPORTS_SUBGROUP_FINDING",
"SubgroupFinding"
],
[
"SubgroupFinding",
"BY_BIOMARKER",
"Biomarker"
],
[
"SubgroupFinding",
"BY_GENOMIC_ALTERATION",
"GenomicAlteration"
],
[
"SubgroupFinding",
"REPORTED_FOR",
"Cohort"
]
]

neo4j and llm config

In [7]:

NEO4J_URI = "neo4j://awsaidnval000z.jnj.com:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "MVP2026!"
TARGET_DB = "newone"


AZURE_OPENAI_KEY="f0e56505e5594c9db822a6e7addc7209"
AZURE_OPENAI_ENDPOINT="https://ims-openai-qa.openai.azure.com/"
AZURE_OPENAI_MODEL="iMS_GPT4o_QA"

LangGraph implementation: route to the right retriever + validate the answer

In [8]:
class GraphState(TypedDict):
        query: str
        tool: str
        answer: str
        raw_result:str
        context:str
        valid:bool
        refined_answer:str

In [ ]:
def router_node(state: GraphState):
 
    query = state["query"]
 
    routing_prompt = f"""
You are a query routing agent.
 
Choose the best retriever.
 
### vector:
Use for conceptual explanations or semantic similarity search.
 
### hybrid:
Use when keyword matching with semantic search is needed.
Good for summaries, tables, and extracting details from text.
 
### vector_cypher:
Use when the query requires graph reasoning such as:
- entity relationships
- comparisons
- aggregations
- structured queries
- multi-hop reasoning
 
Query:
{query}
 
Return ONLY one word:
vector
hybrid
vector_cypher
Reason:<short explanation>
"""
 
    decision = llm.invoke(routing_prompt).content.strip().lower()
 
    print("Router Decision:", decision)
 
    return {"tool": decision}

In [25]:
def vector_node(state: GraphState):
    query=state['query']
    raw_result=vector_rag.retriever.search(query)
   
    ANALYSIS_PROMPT = """
You are a biomedical research expert.
 
Provide a COMPREHENSIVE and VERY DETAILED analytical answer.
 
Include:
- All entities 
- Relationships between entities
- Numerical statistics (%, HR, CI, p-values)
- Comparisons 
- Trends over time
- Evidence-backed reasoning
-Maximum detail with quantitative data
 
Structure output clearly using headings.
"""
    enhanced_query = ANALYSIS_PROMPT + "\n\nUser Question:\n" + query
 

 
    response = vector_rag.search(
    query_text=enhanced_query,
    retriever_config={"top_k": 35}
            )
    print(f"[vector_node] Retrive_answer : {response}")
    
    return {"answer": response.answer,"context":response.context}


In [32]:
def hybrid_node(state: GraphState):
    query=state["query"]
    raw_result=hybrid_rag.retriever.search(query)
    
    
    ANALYSIS_PROMPT = """
You are a biomedical research expert.
 
Provide a COMPREHENSIVE and VERY DETAILED analytical answer.
 
Include:
- All entities 
- Relationships between entities
- Numerical statistics (%, HR, CI, p-values)
- Comparisons 
- Trends over time
- Evidence-backed reasoning
-Maximum detail with quantitative data
 
Structure output clearly using headings.
"""
    enhanced_query = ANALYSIS_PROMPT + "\n\nUser Question:\n" + query
    raw_result=hybrid_rag.retriever.search(query)
    
    
    
    response = hybrid_rag.search(
    query_text=enhanced_query,return_context=True,
    retriever_config={"top_k": 35}
            )
    print(f"[Hybrid_node] Retrive_answer : {response}")
    return {"answer": response.answer,"raw_result":raw_result}

In [58]:
def vector_cypher_node(state: GraphState):

    ANALYSIS_PROMPT = """
You are a biomedical research expert.
 
Provide a COMPREHENSIVE and VERY DETAILED analytical answer.
 
Include:
- All entities 
- Relationships between entities
- Numerical statistics (%, HR, CI, p-values)
- Comparisons 
- Trends over time
- Evidence-backed reasoning
-Maximum detail with quantitative data
 
Structure output clearly using headings.
"""


 
 
    query = state["query"]
    raw_result=vector_cypher_rag.retriever.search(query)
    print("raw_result_vector_cypher:",raw_result)

    enhanced_query = ANALYSIS_PROMPT + "\n\nUser Question:\n" + query
 
    response = vector_cypher_rag.search(
        query_text=query,
        retriever_config={"top_k": 35}
    )
 
    return {
        "answer": response.answer,"raw_result":raw_result
        
    }

In [27]:
def route_decision(state: GraphState):
 
    tool = state["tool"]
 
    if tool == "vector":
        return "vector_node"
 
    elif tool == "vector_cypher":
        return "vector_cypher_node"
 
    else:
        return "hybrid_node"

In [28]:
def validation_node(state: GraphState):
 
    print(" Running Context Utilization Check")
 
    answer = state["answer"]
    raw_result  = state["raw_result"]
 
    validation_prompt = f"""
You are a validation system.
 
Your job is ONLY to check whether the ANSWER uses
all of the  information from the CONTEXT.
 
Rules:
- Extra information in the answer is allowed.
- External reasoning is allowed.
- Additional statistics are allowed.
- Comparative analysis is allowed.
 
You ONLY need to determine:
 
Does the answer use all the  information present in the context?
 
CONTEXT:
{raw_result}
 
ANSWER:
{answer}
 
Return ONLY:
 
Verdict: VALID or INVALID
Reason: detail explanation
"""
 
    result = llm.invoke(validation_prompt).content
 
    print(" Validation Result:", result)
 
    # If INVALID → regenerate answer using reason
    if "INVALID" in result:
 
        print("Regenerating answer using validation feedback")
 
        regenerate_prompt = f"""
The previous answer did not properly use the retrieved context.
 
Context:
{raw_result}
 
Original Answer:
{answer}
 
Validation Feedback:
{result}
 
Rewrite the answer so that it clearly uses information from the context.
Also retrun regenerate answer's verdict: VALID OR INVALID.
"""
 
        new_answer = llm.invoke(regenerate_prompt).content
 
        return {
            "answer": new_answer,
            "validation": result
        }
 
    return {
        "answer": answer,
        "validation": result
    }

In [59]:
async def main():
    global vector_rag,hybrid_rag,vector_cyper_rag,llm
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session(database=TARGET_DB) as session:

        session.run("""
        CREATE FULLTEXT INDEX kg_chunk_fulltext
        IF NOT EXISTS
        FOR (c:Chunk)
        ON EACH [c.text]""")

    create_vector_index(
        driver, INDEX_NAME, label="Chunk",
        embedding_property="embedding",
        dimensions=1536,  # text-embedding-3-large
        similarity_fn="cosine",
        neo4j_database=TARGET_DB
    )
    embedder = AzureOpenAIEmbeddings(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_KEY,
        api_version="2024-02-15-preview",
        model="text-embedding-3-small")

    llm = AzureOpenAILLM(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_KEY,
        model_name=AZURE_OPENAI_MODEL,  
        model_params={"temperature": 0.2,"top_p":0.9,"frequency_penalty":0.2,"presence_penalty":0.3},
        api_version="2024-02-15-preview"

    )

    lc_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2200,
    chunk_overlap=100,
    separators = ["\n\n", "\n", ".", " ", "", "\\","\r\n","\u2022","\x20","\t","\f","\v","\""])
    text_splitter = LangChainTextSplitterAdapter(lc_splitter)
    kg_builder = SimpleKGPipeline(
        llm=llm,
        driver=driver,
        embedder=embedder,
        schema={
            "node_types": node_types,
            "relationship_types": relationship_types,
            "patterns": patterns,
        },
        on_error="IGNORE",
        from_pdf=True,  # True for PDF files
        neo4j_database=TARGET_DB,
        text_splitter=text_splitter

    )
    pdf_file=r"C:\Users\JDas21\GRAPH-RAG\Sample Files\Chen.pdf"
    def graph_exists(driver):
        with driver.session(database=TARGET_DB) as session:
            
            result = session.run("MATCH (n) RETURN count(n) AS count")
            count = result.single()["count"]
            print(f"Database={TARGET_DB} node count:", count)
            return count > 0

        
    if not graph_exists(driver):
        print("Graph not found.Runing ingestion")

        await kg_builder.run_async(file_path=pdf_file)
    else:
        print("Graph already exists.skipping ingestion.")
    
    def extract_json(text):
        """Extract clean JSON from LLM response"""
 
        text = re.sub(r"```json|```", "", text).strip()
 
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
 
        raise ValueError("No valid JSON found")
    
    def get_schema(driver):
        with driver.session(database=TARGET_DB) as session:
            labels = session.run(
            "CALL db.labels() YIELD label RETURN collect(label) as labels"
            ).single()["labels"]
 
            rels = session.run(
            "MATCH ()-[r]->() RETURN collect(DISTINCT type(r)) as rels"
        ).single()["rels"]
 
        return labels, rels
    
    def build_prompt(question, labels, rels, last_error=None, bad_query=None):
    
        base_prompt = f"""
You are an expert Neo4j Cypher generator for a GraphRAG system.
 
=====================
DATABASE SCHEMA
=====================
Node Labels: {labels}
Relationships: {rels}
 
=====================
USER QUESTION
=====================
"{question}"
 
=====================
CORE RULES
=====================
 
1. SCHEMA STRICTNESS
- Use ONLY the provided node labels and relationships
- DO NOT invent new labels or relationships
 
2. DATA UNDERSTANDING
- The graph contains unstructured text in Chunk nodes
- Many answers are stored inside chunk.text (NOT structured nodes)
 
3. QUERY STRATEGY
 
👉 If the question asks for:
- summaries
- results
- findings
- safety, outcomes, conclusions
- numeric values (counts, concentrations, etc.)
 
THEN:
✔ Search inside Chunk.text
✔ DO NOT assume structured nodes exist
 
4. TEXT SEARCH STRATEGY (CRITICAL - MUST FOLLOW)
 
- Break the question into SINGLE keywords only
- DO NOT use long phrases (❌ "through concentration")
- Use ONLY important words:
    ✔ nouns, medical terms, numbers
 
- ALWAYS use OR between conditions
- NEVER use AND for text matching
 
✅ CORRECT:
    toLower(c.text) CONTAINS "plasma"
    OR toLower(c.text) CONTAINS "concentration"
    OR toLower(c.text) CONTAINS "ada"
 
❌ WRONG:
    CONTAINS "plasma" AND CONTAINS "through concentration"
 
- Use 2 to 4 keywords MAX
- Prefer broader words over specific phrases
 
5. FALLBACK STRATEGY (VERY IMPORTANT)
 
- If query might return no results:
    → Use broader keywords
    → Reduce conditions
    → Avoid over-filtering
 
6. GENERAL RULES
- Always start from (c:Chunk)
- Always use toLower(c.text)
- Always include LIMIT 50
- Return c.text
 
=====================
OUTPUT FORMAT (STRICT)
=====================
 
Return ONLY valid JSON:
 
{{
  "query": "valid Cypher query",
  "reasoning": "why this query works"
}}
"""
 
    # Retry prompt (self-healing)
        if last_error:
            return f"""
The previous query failed.
 
Bad Query:
{bad_query}
 
Error:
{last_error}
 
Fix the query using correct schema.
 
{base_prompt}
"""
 
        return base_prompt
    

    def validate_schema_usage(query, labels, rels):
        labels_in_query = re.findall(r":(\w+)", query)
        rels_in_query = re.findall(r":(\w+)\]", query)
 
        label_ok = all(label in labels for label in labels_in_query)
        rel_ok = all(rel in rels for rel in rels_in_query)
 
        return label_ok and rel_ok
    MAX_RETRIES = 3

    import time
 
    def generate_cypher_with_retry(question, driver, llm):
 
        labels, rels = get_schema(driver)
 
        attempt = 0
        last_error = None
        bad_query = None
 
        while attempt < MAX_RETRIES:
 
            print(f"\n🔁 Attempt {attempt+1}")
 
            prompt = build_prompt(question, labels, rels, last_error, bad_query)
 
            response = llm.invoke(prompt)
 
            print("\n🔍 RAW LLM OUTPUT:\n", response.content)
 
        # ✅ Step 1: Extract JSON
            try:
                parsed = extract_json(response.content)
                query = parsed["query"]
                reasoning = parsed.get("reasoning", "")
            except Exception:
                print("❌ JSON parsing failed")
                last_error = "Invalid JSON format"
                attempt += 1
                continue
 
            print("🧠 Reasoning:", reasoning)
            print("📌 Query:", query)
 
        # ✅ Step 2: Schema validation
            if not validate_schema_usage(query, labels, rels):
                print("❌ Schema validation failed")
                last_error = "Query uses invalid schema"
                bad_query = query
                attempt += 1
                continue
 
        # ✅ Step 3: Cypher validation
            try:
                with driver.session(database=TARGET_DB) as session:
                    session.run("EXPLAIN " + query)
 
                print("✅ Query valid")
 
                return {
                "query": query,
                "reasoning": reasoning,
                "attempts": attempt + 1
            }
 
            except Exception as e:
                print("❌ Query failed:", str(e))
                last_error = str(e)
                bad_query = query
                attempt += 1
                time.sleep(1)
 
    # 🚨 Fallback
        print("🚨 Using fallback query")
 
        fallback_query = """
    MATCH (chunk:Chunk)-[*1..2]->(n)
    RETURN chunk.text, labels(n), properties(n)
    """
 
        return {
        "query": fallback_query,
        "reasoning": "Fallback due to repeated failures",
        "attempts": MAX_RETRIES
    }

    def run_query(driver, query):
        with driver.session(database=TARGET_DB) as session:
            result = session.run(query)
            return [record.data() for record in result]
        
    def graph_rag_query(question, driver, llm):
 
        result = generate_cypher_with_retry(question, driver, llm)
        query=result['query']
 
        print("\n📊 FINAL QUERY:", result["query"])
 
        records = run_query(driver, result["query"])
 
        print("\n📊 RESULTS:", records[:5])
 
        return {"query":query}
    
    result=graph_rag_query("For subjects who first became ADA+ at week 16, what is their mean Trough at Week 36?", driver, llm)


    








    

    
    

    
    retriever_vector = VectorRetriever(driver, INDEX_NAME, embedder, neo4j_database=TARGET_DB)
    retriever_hybrid=HybridRetriever(driver=driver,neo4j_database=TARGET_DB,vector_index_name=INDEX_NAME,embedder=embedder,fulltext_index_name='kg_chunk_fulltext')
    vector_rag = GraphRAG(retriever=retriever_vector, llm=llm)
    hybrid_rag=GraphRAG(retriever=retriever_hybrid,llm=llm)

    cypher_query="""MATCH (chunk:Chunk)
 
    OPTIONAL MATCH (chunk)<-[:FROM_CHUNK]-(entity)
 
    OPTIONAL MATCH (entity)-[r]->(related)
 
    OPTIONAL MATCH (chunk)-[:NEXT_CHUNK]->(next_chunk)
 
    OPTIONAL MATCH (chunk)-[:FROM_DOCUMENT]->(doc)
 
    WITH chunk,
    collect(DISTINCT coalesce(entity.name, entity.id, entity.text)) AS entities,
    collect(DISTINCT type(r)) AS relationships,
    collect(DISTINCT coalesce(related.name, related.id, related.text)) AS related_entities,
    next_chunk,
    doc
 
    RETURN
    chunk.text AS text,
    entities,
    relationships,
    related_entities,
    next_chunk.text AS next_chunk,
    doc.name AS document
    LIMIT 25"""
    retriever_vectorcyper=VectorCypherRetriever(
    driver=driver,
    index_name="chunk_embeddings",
    embedder=embedder,
    retrieval_query=result['query'],
    neo4j_database=TARGET_DB
)
    vector_cypher_rag=GraphRAG(retriever=retriever_vectorcyper,llm=llm)
    workflow = StateGraph(GraphState)
    workflow.add_node("router", router_node)
    workflow.add_node("vector_node", vector_node)
    workflow.add_node("hybrid_node", hybrid_node)
    workflow.add_node("vector_cypher_node",vector_cypher_node)

    workflow.add_node("validation_node", validation_node)
    workflow.add_edge("vector_node", "validation_node")
    workflow.add_edge("hybrid_node", "validation_node")
    workflow.add_edge("vector_cypher_node","validation_node")
    workflow.add_edge("validation_node", END)

    workflow.set_entry_point("router")
    workflow.add_conditional_edges(
    "router",
    route_decision,
    {
        "vector_node": "vector_node",
        "hybrid_node": "hybrid_node",
        "vector_cypher_node":"vector_cypher_node"
        
    },
)
    app = workflow.compile()
    query = "For subjects who first became ADA+ at week 16, what is their mean Trough at Week 36?"
    result = app.invoke({"query": query,"tool": ""})
    print(result["answer"])
    #print(app.get_graph().draw_ascii())

    driver.close()
await main()
    





Database=newone node count: 5470
Graph already exists.skipping ingestion.

🔁 Attempt 1

🔍 RAW LLM OUTPUT:
 {
  "query": "MATCH (c:Chunk)\nWHERE toLower(c.text) CONTAINS \"ada\"\n  OR toLower(c.text) CONTAINS \"antidrug\"\n  OR toLower(c.text) CONTAINS \"trough\"\n  OR toLower(c.text) CONTAINS \"week 36\"\nRETURN c.text\nLIMIT 50",
  "reasoning": "The question is about subjects who first became ADA+ at week 16 and their mean trough at week 36, which is a numeric outcome/result. Per the rules, such information is likely embedded in unstructured text within Chunk nodes. The query starts from (c:Chunk) and searches broadly in c.text for key single-word/short tokens related to the concept: 'ada' (for ADA+), 'antidrug' (for antidrug antibodies), 'trough' (for trough concentration), and 'week 36' (timepoint of interest). These are combined with OR to avoid over-filtering and increase recall. The query returns up to 50 matching chunk texts, as required."
}
🧠 Reasoning: The question is about su

In [34]:
from IPython.display import display, HTML
 
display(HTML("<style>.output_scroll { height: auto !important; }</style>"))